In [ ]:
import copy
import os
import sys
import warnings
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
from tqdm.notebook import tqdm

import torch
import timm
import torchvision.transforms as transforms

from sklearn.calibration import CalibratedClassifierCV
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    StratifiedKFold, cross_val_score,
    train_test_split
)
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.ensemble import VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression as SklearnLR
from sklearn.svm import SVC as SklearnSVC
from xgboost import XGBClassifier

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# GPU-accelerated models (cuML) with sklearn fallback
try:
    from cuml.linear_model import LogisticRegression
    from cuml.svm import SVC
    CUML = True
    print("Using cuML (GPU) for Logistic Regression and SVM")
except ImportError:
    from sklearn.linear_model import LogisticRegression
    from sklearn.svm import SVC
    CUML = False
    print("cuML not found, falling back to sklearn (CPU)")

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    OPTUNA_OK = True
except ImportError:
    OPTUNA_OK = False
    print("Optuna not found, will use fixed SVM hyperparameters")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Torch device: {DEVICE}")

In [ ]:
import sys
from pathlib import Path

# Check if we are running inside Google Colab
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    print("☁️ Running on Google Colab. Setting up Drive paths...")

    drive.mount("/content/drive")

    ZIP_PATH = Path("/content/drive/MyDrive/animals.zip")
    EXTRACT_DIR = Path("/content/data")

    checkpoint_1 = EXTRACT_DIR / "train_metadata.csv"
    checkpoint_2 = EXTRACT_DIR / "animals" / "train_metadata.csv"

    if checkpoint_1.exists() or checkpoint_2.exists():
        print("Data already unzipped.")
    else:
        if ZIP_PATH.exists():
            print("Unzipping now...")
            os.system(f"unzip -q {ZIP_PATH} -d {EXTRACT_DIR}")
            print("Unzipping complete!")
        else:
            print("Could not find 'animals.zip' in the root of your Google Drive.")
else:
    print("Running locally. Skipping Google Drive & unzip setup.")

In [ ]:
# Configuration — auto-adapts based on detected environment

IN_COLAB = "google.colab" in sys.modules
HAS_CUDA = torch.cuda.is_available()

if HAS_CUDA:
    # Local GPU or Colab with GPU runtime
    GPU_NAME = torch.cuda.get_device_name(0)
    print(f"GPU detected: {GPU_NAME}")
    BATCH_SIZE = 64
    TTA_CROPS = 5
    PCA_DIMS = 256
    XGB_TREES = 500
    OPTUNA_TRIALS = 40
elif IN_COLAB:
    # Colab CPU runtime
    print("Running on Colab CPU")
    BATCH_SIZE = 32
    TTA_CROPS = 3
    PCA_DIMS = 128
    XGB_TREES = 200
    OPTUNA_TRIALS = 15
else:
    # Local CPU
    print("Running on local CPU")
    BATCH_SIZE = 32
    TTA_CROPS = 1
    PCA_DIMS = 128
    XGB_TREES = 100
    OPTUNA_TRIALS = 10

if IN_COLAB:
    EXTRACT_DIR = Path("/content/data")
    BASE_DIR = (EXTRACT_DIR / "animals") if (EXTRACT_DIR / "animals").exists() else EXTRACT_DIR
else:
    BASE_DIR = Path("data/animals")

TRAIN_META = BASE_DIR / "train_metadata.csv"
TEST_META = BASE_DIR / "test_metadata.csv"
COLOR_HIST = BASE_DIR / "color_histogram.csv"
HOG_PCA = BASE_DIR / "hog_pca.csv"
ADDITIONAL_FEATS = BASE_DIR / "additional_features.csv"
SUBMISSION_PATH = BASE_DIR / "submission.csv"
FIGURES_DIR = BASE_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"BASE_DIR    : {BASE_DIR}")
print(f"BATCH_SIZE  : {BATCH_SIZE}")
print(f"TTA_CROPS   : {TTA_CROPS}")
print(f"PCA_DIMS    : {PCA_DIMS}")
print(f"XGB_TREES   : {XGB_TREES}")
print(f"OPTUNA_TRIALS: {OPTUNA_TRIALS}")

In [ ]:
# Load CSV features

print("\n--- Loading CSV features ---")

train_meta = pd.read_csv(TRAIN_META)
test_meta = pd.read_csv(TEST_META)

color = pd.read_csv(COLOR_HIST)
hog = pd.read_csv(HOG_PCA)
add = pd.read_csv(ADDITIONAL_FEATS)

def merge_features(meta, *feature_dfs):
    df = meta.copy()
    for fdf in feature_dfs:
        df = df.merge(fdf, on="image_id", how="left")
    return df

train_df = merge_features(train_meta, color, hog, add)
test_df = merge_features(test_meta, color, hog, add)

meta_cols = ["image_id", "image_path", "class_id", "class_name"]
csv_feature_cols = [c for c in train_df.columns if c not in meta_cols]

class_names = (
    train_df[["class_id", "class_name"]]
    .drop_duplicates()
    .sort_values("class_id")["class_name"]
    .tolist()
)

print(f"Train samples: {len(train_df)},  Test samples: {len(test_df)}")
print(f"CSV feature dims: {len(csv_feature_cols)}")
print(f"Classes: {class_names}")

In [ ]:
def make_preprocess():
    return transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])

def tta_crops(img, n):
    """
    Return n augmented tensors for test-time averaging.
    n=1 returns only the centre crop (no TTA).
    n=5 returns four corners + centre.
    """
    preprocess = make_preprocess()
    if n <= 1:
        return [preprocess(img)]
    big = img.resize((256, 256), Image.BILINEAR)
    crops = [
        big.crop((0, 0, 224, 224)),
        big.crop((32, 0, 256, 224)),
        big.crop((0, 32, 224, 256)),
        big.crop((32, 32, 256, 256)),
        big.crop((16, 16, 240, 240)),
    ]
    return [preprocess(c) for c in crops[:n]]

def extract_embeddings(image_paths, base_dir, cache_path, model_name, batch_size=32):
    """
    Extract embeddings from any timm model.
    Uses a disk cache so re-running the notebook is fast.
    TTA averaging is applied per image before caching.
    """
    if cache_path.exists():
        print(f"  Loading cached embeddings from {cache_path.name}")
        return np.load(cache_path)

    print(f"  Extracting {model_name} embeddings for {len(image_paths)} images (TTA={TTA_CROPS})...")

    model = timm.create_model(model_name, pretrained=True, num_classes=0)
    model.eval().to(DEVICE)

    all_embeddings = []
    for i in tqdm(range(0, len(image_paths), batch_size), desc=f"  {model_name}"):
        batch_paths = image_paths[i: i + batch_size]
        crop_tensors = []
        for rel_path in batch_paths:
            try:
                img = Image.open(base_dir / rel_path).convert("RGB")
            except Exception:
                img = Image.new("RGB", (64, 64), (128, 128, 128))
            crop_tensors.append(tta_crops(img, TTA_CROPS))

        # crop_tensors: list(B) of list(n_crops) → average across crops
        n = len(crop_tensors[0])
        crop_embs = []
        for c in range(n):
            batch = torch.stack([crop_tensors[b][c] for b in range(len(batch_paths))]).to(DEVICE)
            with torch.no_grad():
                crop_embs.append(model(batch).cpu().numpy())
        all_embeddings.append(np.mean(np.stack(crop_embs), axis=0))

    embeddings = np.vstack(all_embeddings)
    np.save(cache_path, embeddings)
    print(f"  Saved to {cache_path.name}")
    return embeddings

In [ ]:
# Extract embeddings: ViT-Base/16 + ResNet-50 + EfficientNet-B3
# ViT-Base: 768-dim attention-based features
# ResNet-50: 2048-dim CNN features, complements ViT
# EfficientNet-B3: 1536-dim, strong on lower-resolution upscaled inputs

BACKBONES = [
    ("vit_base_patch16_224", "vit_base"),
    ("resnet50", "resnet50"),
    ("efficientnet_b3", "effnet_b3"),
]

backbone_train = {}
backbone_test = {}

for model_name, stem in BACKBONES:
    suffix = f"_tta{TTA_CROPS}.npy"
    print(f"\n--- {model_name} ---")
    backbone_train[stem] = extract_embeddings(
        train_df["image_path"].values, BASE_DIR,
        BASE_DIR / f"train_{stem}{suffix}", model_name, BATCH_SIZE)
    backbone_test[stem] = extract_embeddings(
        test_df["image_path"].values, BASE_DIR,
        BASE_DIR / f"test_{stem}{suffix}", model_name, BATCH_SIZE)
    print(f"  Shape: {backbone_train[stem].shape}")

In [ ]:
# Combine and normalise features
# Per-backbone PCA compresses noise and speeds up SVM fitting

print("\n--- Combining and normalising features ---")

X_train_csv = np.nan_to_num(train_df[csv_feature_cols].values.astype(np.float32))
X_test_csv = np.nan_to_num(test_df[csv_feature_cols].values.astype(np.float32))
scaler_csv = StandardScaler()
X_train_csv = scaler_csv.fit_transform(X_train_csv)
X_test_csv = scaler_csv.transform(X_test_csv)

parts_train = []
parts_test = []

for stem in [s for _, s in BACKBONES]:
    tr = backbone_train[stem]
    te = backbone_test[stem]

    sc = StandardScaler()
    tr = sc.fit_transform(tr)
    te = sc.transform(te)

    if PCA_DIMS > 0 and tr.shape[1] > PCA_DIMS:
        pca = PCA(n_components=PCA_DIMS, random_state=42)
        tr = pca.fit_transform(tr)
        te = pca.transform(te)
        print(f"  {stem}: PCA {backbone_train[stem].shape[1]} -> {PCA_DIMS}d  "
              f"(var kept: {pca.explained_variance_ratio_.sum():.3f})")

    parts_train.append(tr)
    parts_test.append(te)

parts_train.append(X_train_csv)
parts_test.append(X_test_csv)

X_train = np.hstack(parts_train).astype(np.float32)
X_test = np.hstack(parts_test).astype(np.float32)
y_train = train_df["class_id"].values

print(f"\nTotal feature dims: {X_train.shape[1]}")
print(f"  Train: {X_train.shape}")
print(f"  Test:  {X_test.shape}")

In [ ]:
# Cross-validation helper

def cv_evaluate(model, X, y, n_splits=5, label="Model"):
    """Stratified k-fold cross-validation."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    scores = cross_val_score(model, X, y, cv=skf, scoring="accuracy", n_jobs=-1)
    print(f"  {label}: {scores.mean():.4f} ± {scores.std():.4f}  (folds: {np.round(scores, 4)})")
    return scores.mean()

In [ ]:
print("\n--- SVM Hyperparameter Search ---")

is_cuml = LogisticRegression.__module__.startswith("cuml")

if OPTUNA_OK:
    print(f"Running Optuna ({OPTUNA_TRIALS} trials)...")

    def objective(trial):
        C = trial.suggest_float("C", 1e0, 1e3, log=True)
        gamma = trial.suggest_float("gamma", 1e-4, 1e-1, log=True)
        clf = SklearnSVC(kernel="rbf", C=C, gamma=gamma, random_state=42)
        skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
        return cross_val_score(clf, X_train, y_train, cv=skf,
                               scoring="accuracy", n_jobs=-1).mean()

    study = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)
    SVM_C = study.best_params["C"]
    SVM_GAMMA = study.best_params["gamma"]
    print(f"  Best params: C={SVM_C:.4f}, gamma={SVM_GAMMA:.6f}  (CV={study.best_value:.4f})")
else:
    from scipy.stats import loguniform
    from sklearn.model_selection import RandomizedSearchCV

    n_iter = 15 if is_cuml else 5
    svc_kwargs = {"kernel": "rbf", "probability": False, "random_state": 42}
    param_dist = {
        "C": loguniform(1, 1000),
        "gamma": loguniform(1e-4, 1e-1),
    }
    svm_search = RandomizedSearchCV(
        SklearnSVC(**svc_kwargs),
        param_distributions=param_dist,
        n_iter=n_iter,
        cv=3,
        scoring="accuracy",
        n_jobs=-1,
        random_state=42,
        verbose=3
    )
    svm_search.fit(X_train, y_train)
    SVM_C = svm_search.best_params_["C"]
    SVM_GAMMA = svm_search.best_params_["gamma"]
    print(f"  Best params: {svm_search.best_params_}")
    print(f"  Best CV score: {svm_search.best_score_:.4f}")

In [ ]:
print("\n--- Model 1: Logistic Regression ---")

is_cuml = LogisticRegression.__module__.startswith("cuml")
solver_name = "qn" if is_cuml else "lbfgs"

lr = LogisticRegression(
    C=10.0,
    max_iter=2000,
    solver=solver_name,
    class_weight="balanced"
)

lr_cv = cv_evaluate(lr, X_train, y_train, label="Logistic Regression")

In [ ]:
print("\n--- Model 2: SVM (RBF, calibrated) ---")

# CalibratedClassifierCV gives better probability estimates for soft-voting
# cuML SVM already supports probability=True natively
is_cuml = SVC.__module__.startswith("cuml")

if is_cuml:
    svm = SVC(kernel="rbf", probability=True, C=SVM_C, gamma=SVM_GAMMA)
else:
    base_svm = SklearnSVC(kernel="rbf", C=SVM_C, gamma=SVM_GAMMA, random_state=42)
    svm = CalibratedClassifierCV(base_svm, cv=3, method="sigmoid")

svm_cv = cv_evaluate(svm, X_train, y_train, label="SVM RBF (calibrated)")

In [ ]:
print("\n--- Model 3: XGBoost ---")

if DEVICE == "cuda":
    print("Running XGBoost on GPU (CUDA)")
else:
    print("Running XGBoost on CPU")

xgb = XGBClassifier(
    n_estimators=XGB_TREES,
    max_depth=6,
    learning_rate=0.04,
    subsample=0.8,
    colsample_bytree=0.7,
    min_child_weight=3,
    reg_alpha=0.1,
    reg_lambda=1.0,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
    device=DEVICE
)

xgb_cv = cv_evaluate(xgb, X_train, y_train, label="XGBoost")

In [ ]:
print("\n--- Model 4: Soft Voting Ensemble ---")

is_cuml = LogisticRegression.__module__.startswith("cuml")
solver_name = "qn" if is_cuml else "lbfgs"

if is_cuml:
    lr_inst = LogisticRegression(C=10.0, max_iter=2000, solver="qn", class_weight="balanced")
    svm_inst = SVC(kernel="rbf", probability=True, C=SVM_C, gamma=SVM_GAMMA)
else:
    lr_inst = LogisticRegression(C=10.0, max_iter=2000, solver="lbfgs", class_weight="balanced")
    base_svm_inst = SklearnSVC(kernel="rbf", C=SVM_C, gamma=SVM_GAMMA, random_state=42)
    svm_inst = CalibratedClassifierCV(base_svm_inst, cv=3, method="sigmoid")

soft_vote = VotingClassifier(
    estimators=[
        ("lr", lr_inst),
        ("svm", svm_inst),
        ("xgb", XGBClassifier(
            n_estimators=XGB_TREES,
            max_depth=6,
            learning_rate=0.04,
            subsample=0.8,
            colsample_bytree=0.7,
            min_child_weight=3,
            reg_alpha=0.1,
            reg_lambda=1.0,
            eval_metric="mlogloss",
            random_state=42,
            n_jobs=-1,
            device=DEVICE
        )),
    ],
    voting="soft"
)

soft_vote_cv = cv_evaluate(soft_vote, X_train, y_train, label="Soft Voting Ensemble")

In [ ]:
print("\n--- Model 5: Stacking Ensemble ---")

# Stacking trains a meta-learner on out-of-fold predictions from base models
# This lets the meta-learner learn which base model to trust per input region

is_cuml = LogisticRegression.__module__.startswith("cuml")

if is_cuml:
    lr_s = LogisticRegression(C=10.0, max_iter=2000, solver="qn", class_weight="balanced")
    svm_s = SVC(kernel="rbf", probability=True, C=SVM_C, gamma=SVM_GAMMA)
else:
    lr_s = LogisticRegression(C=10.0, max_iter=2000, solver="lbfgs", class_weight="balanced")
    base_svm_s = SklearnSVC(kernel="rbf", C=SVM_C, gamma=SVM_GAMMA, random_state=42)
    svm_s = CalibratedClassifierCV(base_svm_s, cv=3, method="sigmoid")

stacking = StackingClassifier(
    estimators=[
        ("lr", lr_s),
        ("svm", svm_s),
        ("xgb", XGBClassifier(
            n_estimators=XGB_TREES,
            max_depth=6,
            learning_rate=0.04,
            subsample=0.8,
            colsample_bytree=0.7,
            min_child_weight=3,
            reg_alpha=0.1,
            reg_lambda=1.0,
            eval_metric="mlogloss",
            random_state=42,
            n_jobs=-1,
            device=DEVICE
        )),
    ],
    final_estimator=SklearnLR(C=1.0, max_iter=1000, random_state=42),
    cv=5,
    stack_method="predict_proba",
    n_jobs=-1
)

stacking_cv = cv_evaluate(stacking, X_train, y_train, label="Stacking Ensemble")

In [ ]:
print("\n--- Cross-validation summary ---")

results = {
    "Logistic Regression": lr_cv,
    "SVM RBF (calibrated)": svm_cv,
    "XGBoost": xgb_cv,
    "Soft Voting Ensemble": soft_vote_cv,
    "Stacking Ensemble": stacking_cv,
}
for name, score in sorted(results.items(), key=lambda x: -x[1]):
    print(f"  {name}: {score:.4f}")

best_name = max(results, key=results.get)
model_map = {
    "Logistic Regression": lr,
    "SVM RBF (calibrated)": svm,
    "XGBoost": xgb,
    "Soft Voting Ensemble": soft_vote,
    "Stacking Ensemble": stacking,
}
best_model = model_map[best_name]
print(f"\nBest model: {best_name} ({results[best_name]:.4f})")

print(f"Fitting {best_name} on full training set...")
best_model.fit(X_train, y_train)
train_preds = best_model.predict(X_train)
print(f"Training accuracy (sanity check): {accuracy_score(y_train, train_preds):.4f}")

In [ ]:
print("\n--- Generating confusion matrix on 20% holdout ---")

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=42
)

eval_model = copy.deepcopy(best_model)
eval_model.fit(X_tr, y_tr)
val_preds = eval_model.predict(X_val)
val_acc = accuracy_score(y_val, val_preds)
print(f"Holdout validation accuracy: {val_acc:.4f}")

cm = confusion_matrix(y_val, val_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.title(f"Confusion Matrix — {best_name} (val acc: {val_acc:.3f})")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()

safe_model_name = best_name.lower().replace(" ", "_").replace("(", "").replace(")", "").replace("-", "_")
plt.savefig(FIGURES_DIR / f"{safe_model_name}.png", dpi=150)
plt.show()
print(f"Saved: {FIGURES_DIR / f'{safe_model_name}.png'}")

print("\nClassification Report:")
print(classification_report(y_val, val_preds, target_names=class_names))

In [ ]:
print("\n--- Generating test predictions ---")

test_preds = best_model.predict(X_test)
submission = pd.DataFrame({
    "image_id": test_meta["image_id"],
    "class_id": test_preds,
})
submission.to_csv(SUBMISSION_PATH, index=False)
print(f"Saved: {SUBMISSION_PATH}  ({len(submission)} rows)")